In [ ]:
import cv2
import numpy as np

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100)
    )

    for (x, y, w, h) in faces:

        # Face box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        # Center face region (better for skin analysis)
        roi_x1 = x + int(w * 0.30)
        roi_y1 = y + int(h * 0.30)

        roi_x2 = x + int(w * 0.70)
        roi_y2 = y + int(h * 0.70)

        skin_roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]

        # Blue box showing sampled skin area
        cv2.rectangle(
            frame,
            (roi_x1, roi_y1),
            (roi_x2, roi_y2),
            (255, 0, 0),
            2
        )

        if skin_roi.size > 0:

            # Convert to LAB
            lab = cv2.cvtColor(skin_roi, cv2.COLOR_BGR2LAB)

            # Average lightness
            L = np.mean(lab[:, :, 0])

           if L > 200:
    tone = "Fair"
elif L >= 170:
    tone = "Light"
elif L >= 140:
    tone = "Medium"
elif L >= 100:
    tone = "Olive"
else:
    tone = "Deep"

            # Display category
            cv2.putText(
                frame,
                f"Tone: {tone}",
                (x, y - 15),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            # Display L value
            cv2.putText(
                frame,
                f"L Value: {L:.1f}",
                (x, y + h + 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )

    cv2.imshow("Skin Tone Detection", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()